# Generazione e test del ritiro della transitività

La pipeline prepara coppie di snapshot Turtle. Nel secondo viene rimossa **soltanto**
la tripla `(proprietà bersaglio, rdf:type, owl:TransitiveProperty)`: i fatti e la dichiarazione
`owl:ObjectProperty` restano invariati. La query riguarda una relazione tra individui.

Le funzioni sono implementate in `transitive_property_benchmark.py` e importate sotto.
**Le celle di configurazione e creazione dell’adattatore non chiamano modelli.**
La cella di preparazione, ora attiva, avvia Ollama e HermiT quando viene eseguita.
Gli esempi di valutazione restano commentati. Non usare Run All per la sola ispezione.
Java e `owlready2` servono quando si esegue la verifica formale; nessun modello viene scaricato.

Il protocollo mantiene due turni per caso, indipendenti dagli altri casi: ontologia originale,
poi ontologia aggiornata con la risposta autentica precedente. Il primo output non viene
corretto né sostituito con la risposta di riferimento.


## Configurazione e funzioni

`GenerationConfig` descrive ciò che chiediamo al generatore. I metadati `observed` descrivono
ciò che è stato misurato nel grafo: numero di triple, individui e caratteristiche OWL.
`path_length` conta i collegamenti del percorso; il supporto inferenziale contiene anche
la dichiarazione di transitività.

Il generatore procedurale usa nomi simbolici, catene separate e distrattori su un'altra proprietà.
`domain` è solo un'etichetta di provenienza in questa modalità. Le ramificazioni e i percorsi
alternativi sono possibili estensioni, non parametri già implementati.


In [1]:
from pathlib import Path
import sys
from dataclasses import asdict

roots = [Path.cwd(), Path.cwd() / "neurosymbolic-epistemic", Path.cwd().parent]
ROOT = next((p.resolve() for p in roots if (p / "transitive_property_benchmark.py").exists()), None)
if ROOT is None:
    raise FileNotFoundError("Aprire il notebook dalla repository.")
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from transitive_property_benchmark import (
    GenerationConfig, PropertyOracle, OPTIONS,
    generate_onto, make_ollama_generator, validate_onto, add_individuals_and_facts,
    select_candidate_queries, withdraw_transitivity, verify_case,
    prepare_cases, save_dataset, load_dataset,
    build_messages, run_case, evaluate_case, evaluate, run_benchmark,
)

list_of_domains = ["only_symbolic"]
CONFIG = GenerationConfig(
    generator="llm", domain="food", seed=42,
    path_length=3, n_distractor_triples=10, n_queries=1,
    max_generation_attempts=3, direct_assertion_control=False,
)
DATASET_DIR = ROOT / "benchmarks" / "transitive_withdrawal_v1_pilot"
MODEL = "gemma4:e4b"  # nome esatto di un modello locale; non viene chiamato qui
LLM_OPTIONS = dict(OPTIONS)  # seed del modello distinto dal seed di generazione
print("Configurazione:", asdict(CONFIG))
print("Destinazione futura:", DATASET_DIR)
print("Nessuna generazione o chiamata al modello avviata.")


Configurazione: {'generator': 'llm', 'domain': 'food', 'seed': 42, 'path_length': 3, 'n_distractor_triples': 10, 'n_queries': 1, 'max_generation_attempts': 3, 'direct_assertion_control': False}
Destinazione futura: /Users/emanuele/Desktop/neurosymbolic-epistemic/neurosymbolic-epistemic/benchmarks/transitive_withdrawal_v1_pilot
Nessuna generazione o chiamata al modello avviata.


In [2]:
# Configurazione dell'adattatore: questa cella NON chiama il modello.
GENERATOR_MODEL = "gemma4:e4b"  # nome esatto del modello locale che genera ontologie
GENERATOR_OPTIONS = {"temperature": 0.6, "num_ctx": 32768, "num_predict": 4096}
LLM_GENERATOR = make_ollama_generator(
    GENERATOR_MODEL, options=GENERATOR_OPTIONS, timeout_s=600,
)
LLM_CONFIG = CONFIG  # usa la configurazione principale (dominio food)

# ESEMPIO DI GENERAZIONE DELLA SOLA ONTOLOGIA — non attivato.
# onto_obj = generate_onto(LLM_CONFIG, llm_generator=LLM_GENERATOR)
# print(onto_obj["generated_onto"])

# ESEMPIO DI PREPARAZIONE COMPLETA — non attivato.
# data, snapshots = prepare_cases(
#     [LLM_CONFIG], PropertyOracle(timeout_s=120), llm_generator=LLM_GENERATOR,
# )
# save_dataset(data, snapshots, ROOT / "benchmarks" / "transitive_withdrawal_v1_llm_pilot")


## Preparazione e validazione dei casi

Le responsabilità sono separate:

| Funzione | Responsabilità |
|---|---|
| `generate_onto` | Ontologia di base, proprietà bersaglio e provenienza. |
| `validate_onto` | Parsing, import, dichiarazioni, consistenza e metadati. Restituisce un rapporto; non rigenera da sola. |
| `add_individuals_and_facts` | Inserisce individui nuovi, catene e fatti distrattori. |
| `select_candidate_queries` | Seleziona query strutturate e trova un supporto candidato con il percorso richiesto. |
| `withdraw_transitivity` | Copia il grafo ed elimina esattamente la dichiarazione della proprietà scelta. |
| `verify_case` | Verifica i due snapshot, l'esito prima/dopo e la sufficienza del supporto di riferimento. |
| `prepare_cases` | Coordina i passaggi con tentativi limitati, log degli scarti e identificatori di ontologia. |
| `save_dataset` / `load_dataset` | Congelano un dataset nuovo; verificano hash e differenze tra snapshot al caricamento. |

`PropertyOracle` usa HermiT: una relazione q è implicata se l'ontologia è consistente e
l'aggiunta della negazione esplicita di q la rende inconsistente. Ogni verifica usa un processo
isolato con timeout. Un errore o timeout **non** diventa una risposta `not_entailed`.

Per i casi di ritiro si accettano solo query `entailed → not_entailed`.
Con `direct_assertion_control=True` il collegamento è asserito direttamente e si richiede
`entailed → entailed`, pur rimuovendo la stessa dichiarazione. In questi controlli la lunghezza
osservata del percorso minimo è 1, anche se la catena generata è più lunga.

Il formato delle prove attuale supporta triple con IRI HTTP(S), senza blank node.
Le ontologie LLM con strutture anonime vengono scartate; la validazione non sostituisce un
controllo completo del profilo OWL 2 DL. La prima modalità procedurale evita questi costrutti.


In [3]:
# PREPARAZIONE ESPLICITA: eseguire questa cella avvia Ollama e HermiT.
oracle = PropertyOracle(timeout_s=120)
configs = [CONFIG]
llm_generator = LLM_GENERATOR
data, snapshots = prepare_cases(configs, oracle, llm_generator=llm_generator)
print("DATA : ", data, "\n snapshot: ", snapshots)
# save_dataset(data, snapshots, DATASET_DIR)  # errore se la cartella esiste già
# print("Casi verificati:", len(data["cases"]))

# ESEMPIO DI AMPLIAMENTO — configurazione di 90 ontologie, una query ciascuna.
# configs = [
#     GenerationConfig(seed=1000 + 30 * index + repetition,
#                      path_length=depth, n_distractor_triples=10, n_queries=1)
#     for index, depth in enumerate((2, 3, 4))
#     for repetition in range(30)
# ]
# Non aggiungere due volte la stessa configurazione: i grafi duplicati sono rifiutati.


generating ontology using llm
LLM generator inputs:
{
  "model": "gemma4:e4b",
  "messages": [
    {
      "role": "user",
      "content": "Generate a consistent OWL ontology in Turtle for domain 'food'. Use absolute HTTP(S) IRIs. Return a JSON object with generated_onto (a Turtle string) and target_property (an absolute IRI). Include a declared object property typed owl:TransitiveProperty; identify it using target_property in the returned object. Use only named entities: no blank nodes, anonymous restrictions, RDF lists or imports. Other named classes and properties may provide domain context. Do not materialize inferred facts. Individuals, benchmark chains, distractor facts and queries will be inserted by a separate builder; do not generate them or answer any benchmark query. For only_symbolic use neutral symbols without domain meanings.\nGENERATION CONFIGURATION:\n{\"direct_assertion_control\": false, \"domain\": \"food\", \"generator\": \"llm\", \"max_generation_attempts\": 3, \"n

KeyboardInterrupt: 

In [3]:
data

{'version': 'transitive_withdrawal_v1',
 'cases': [{'id': 'transitive_0000_000',
   'ontology_id': 'e38fa6e579fcd4b078c275ca592f979f7daff09ad220604fb12b98a54bd124fa',
   'snapshots': {'original': 'ontology_0000_original.ttl',
    'withdrawal': 'ontology_0000_withdrawal.ttl'},
   'query': {'subject': 'https://example.org/case/6df03b78-eaec-ac77-2836-0719b0c1a5bd#q0_n0',
    'property': 'https://example.org/transitivity/6df03b78eaecac77#p',
    'object': 'https://example.org/case/6df03b78-eaec-ac77-2836-0719b0c1a5bd#q0_n3'},
   'explicit_path_length': 3,
   'reference_support': [{'subject': 'https://example.org/case/6df03b78-eaec-ac77-2836-0719b0c1a5bd#q0_n0',
     'predicate': 'https://example.org/transitivity/6df03b78eaecac77#p',
     'object': 'https://example.org/case/6df03b78-eaec-ac77-2836-0719b0c1a5bd#q0_n1'},
    {'subject': 'https://example.org/case/6df03b78-eaec-ac77-2836-0719b0c1a5bd#q0_n1',
     'predicate': 'https://example.org/transitivity/6df03b78eaecac77#p',
     'object'

### Generazione tramite un LLM: adattatore Ollama

`llm_generator` è una funzione che la pipeline chiama per ottenere l'ontologia di base.
`make_ollama_generator` costruisce questa funzione per un modello locale Ollama.
**Creare l'adattatore non contatta Ollama**; soltanto una successiva chiamata a
`generate_onto(..., llm_generator=LLM_GENERATOR)` o `prepare_cases(...)` avvia la generazione.

L'adattatore richiede JSON strutturato con `generated_onto` (Turtle) e `target_property`
(IRI completa). Controlla che il modello sia già installato, il budget e la capacità del contesto,
il limite di output, il formato JSON, la sintassi Turtle e le dichiarazioni della proprietà.
Import e blank node vengono rifiutati. La consistenza e il gold restano responsabilità dell'oracolo.

La provenienza conserva modello/digest, versione Ollama, opzioni effettive, configurazione,
prompt completi, risposta originale e tempi. Il generatore non risponde alle query:
individui, catene, distrattori e query vengono aggiunti dal builder.

Il modello generatore e quello valutato sono configurati separatamente. La temperatura di
**generazione** predefinita è 0.6; quella del **modello valutato** resta in `LLM_OPTIONS`.
Se il seed non è specificato nelle opzioni del generatore, viene usato quello della configurazione.
I retry restano limitati da `max_generation_attempts`: non cambiano automaticamente seed o opzioni.
Una rigenerazione deterministica può quindi ripetere lo stesso errore.


## Valutazione del modello

`build_messages` prepara i prompt; `run_case` esegue i due turni e conserva le richieste,
le risposte integrali, il JSON interpretato, i tempi e gli errori.
Un output iniziale malformato viene conservato nella cronologia se la chiamata è utilizzabile;
errori infrastrutturali o di contesto impediscono il secondo turno, registrato come dipendenza.
Il budget viene controllato senza troncare l'ontologia.

`evaluate_case` verifica correttezza, presenza e sufficienza degli assiomi citati.
La minimalità è separata dal successo; una prova alternativa valida è accettata.
Il gold e il supporto di riferimento non vengono inseriti nei prompt.

`run_benchmark` ricarica e riverifica il dataset, controlla il modello installato e crea un run in:

```text
generated/transitive_property/<timestamp UTC>_<nome modello>_<identificativo>/
```

Il run contiene `manifest.json`, `responses.jsonl`, `evaluation.json` e `snapshots/`.
I record vengono scritti progressivamente; i run precedenti non sono sovrascritti.


In [ ]:
# ESEMPIO DI RUN LLM — non attivato.
# run_directory, result = run_benchmark(
#     MODEL, DATASET_DIR, options=LLM_OPTIONS,
# )
# print("Run:", run_directory)
# print(result["summary"])

# SOLA LETTURA DI UN DATASET ESISTENTE — non chiama Ollama.
# data, graphs, texts, dataset_hash = load_dataset(DATASET_DIR)
# messages = build_messages(data["cases"][0], texts)
# Per riverificare formalmente usare load_dataset(DATASET_DIR, oracle=PropertyOracle()).


## Metriche e limiti

Per ogni coppia G indica successo originale e U successo sullo snapshot aggiornato.
La risposta positiva deve avere supporto presente e sufficiente; una risposta negativa corretta
richiede lista vuota (convenzione di output, non prova di falsità).

| Metrica | Significato |
|---|---|
| `paired_grounded_revision_score` | Media di G × U su **tutte** le coppie. |
| `original_grounded_score` | Quota di originali corretti con supporto valido. |
| `updated_grounded_score` | Quota di secondi turni riusciti, inclusi eventuali controlli. |
| `withdrawal_score` | Quota di secondi turni riusciti sui soli casi che perdono entailment. |
| `preservation_score` | Quota di secondi turni riusciti sui soli controlli che mantengono entailment. |
| `original_answer_accuracy` | Accuracy dell'etichetta originale, indipendentemente dal supporto. |
| `correct_original_without_valid_support_count` | Originali con etichetta corretta e supporto non valido. |
| `withdrawal_stale_citation_count` | Secondi turni che citano almeno un assioma ritirato. |
| `complete_for_comparison` | Assenza di errori infrastrutturali, di contesto e di dipendenza. |

Gli errori di formato restano insuccessi; gli errori infrastrutturali rendono gli score `null`.
Una categoria senza casi ha score `null`, non zero. I conteggi e i dettagli rimangono consultabili.
Gli score sono anche separati per lunghezza del percorso. `ontology_id` permette di riconoscere
le query dello stesso grafo: più query su un'ontologia non equivalgono a prove indipendenti.

Le funzioni sono coperte da test offline con oracolo e trasporto simulati. Questi controllano
il comportamento del codice. La sezione del primo batch Qwen ha inoltre verificato realmente
la generazione Ollama e la preparazione con Java/HermiT; il runner di valutazione a due turni
non è ancora stato eseguito su un modello reale.


<details>
<summary>Nota iniziale della bozza</summary>

```text
TESTARE TRANSITIVE PROPERTY
1- generazione di ontologies con determinate caratteristiche tramite un generatore llm o altro. Domini decisi a priori da una lista.
2- parsing del grafo per verificare se è corretto: altrimenti si torna a generazione del grafo
3- Si genera la domanda (con owlapy/owlready o a mano) entailed
4- Si prepara lo snapshot not entailed: si rimuove il costrutto owl:TransitiveProperty 
4- Si passa al LLM da testare prima onto completa + query entailed, poi onto 




Le ontologie devono essere di diverse lunghezze, diverso numero di triple e complessità e includere varie proprietà. 
Per ogni ontologia ci può essere anche solo una transitive property
Diverse lunghezze del percorso
Tutte in formato ttl per ora
```
</details>


## Primo batch locale: cinque ontologie generate da Qwen

Questa sezione è autonoma: eseguire soltanto le sue celle per generare e salvare il batch.
Usa `qwen3.6:latest` se installato; altrimenti seleziona la versione Qwen più recente tra
quelle disponibili localmente. Non scarica modelli e non esegue i due turni di valutazione LLM.

Cinque ontologie per quattro domini: **food, school, animals, relatives, food**.
Il secondo food usa un seed diverso. Per tutte: percorso di **3 collegamenti**, **1 query**,
**10 relazioni distrattrici** e nessun collegamento diretto di controllo.
Le dichiarazioni degli individui distrattori aggiungono altre triple: 10 è il numero di relazioni,
non il totale di triple RDF del contesto. La dimensione effettiva dello schema LLM è registrata.

Il modello genera il vocabolario e lo schema di dominio; il builder inserisce catena e distrattori
con nomi simbolici. HermiT verifica consistenza, supporto, entailment originale e non-entailment
successivo al ritiro della sola dichiarazione di transitività.

Ogni esecuzione crea una directory nuova in `generated/transitive_property_ontologies/`.
Ogni ontologia viene salvata appena verificata; in caso di errore i risultati già salvati restano
consultabili e `batch.json` segnala il fallimento. Non usare Run All per la sola lettura.


In [ ]:
# Configurazione autonoma: non chiama ancora Ollama.
from pathlib import Path
from datetime import datetime, timezone
import importlib
import json
import os
import re
import sys
import uuid

_qwen_roots = [Path.cwd(), Path.cwd() / "neurosymbolic-epistemic", Path.cwd().parent]
QWEN_BATCH_ROOT = next((p.resolve() for p in _qwen_roots
                        if (p / "transitive_property_benchmark.py").exists()), None)
if QWEN_BATCH_ROOT is None:
    raise FileNotFoundError("Aprire il notebook dalla repository.")
if str(QWEN_BATCH_ROOT) not in sys.path:
    sys.path.insert(0, str(QWEN_BATCH_ROOT))
import transitive_property_benchmark as transitive_batch
transitive_batch = importlib.reload(transitive_batch)

QWEN_PREFERRED_MODEL = "qwen3.6:latest"
QWEN_BATCH_DOMAINS = ["food", "school", "animals", "relatives", "food"]
QWEN_BATCH_OPTIONS = {"temperature": 0.6, "num_ctx": 32768, "num_predict": 4096}
QWEN_BATCH_CONFIGS = [
    transitive_batch.GenerationConfig(
        generator="llm", domain=domain, seed=20261001 + index,
        path_length=3, n_distractor_triples=10, n_queries=1,
        max_generation_attempts=3, direct_assertion_control=False,
    )
    for index, domain in enumerate(QWEN_BATCH_DOMAINS)
]
QWEN_BATCH_SYSTEM_PROMPT = (
    "Create a compact domain-specific OWL schema, using meaningful names for its domain. "
    "Use exactly four named classes and three object properties. Exactly one property "
    "must be explicitly transitive and it must be the returned target_property. "
    "Use only declarations, named subclass axioms, named domain/range axioms and labels. "
    "Use a named ontology header, for example <https://example.org/domain/ontology> a owl:Ontology. "
    "Never use [], (), _: identifiers, owl:Restriction or OWL lists anywhere in the Turtle. "
    "Every subject must be a named IRI. No individuals, disjointness, property hierarchies, inverses, property chains, "
    "equivalences, blank nodes or anonymous class expressions. Do not answer any query. "
    "Return only the JSON object required by the schema, with Turtle in generated_onto."
)


In [ ]:
# ESECUZIONE ESPLICITA: genera le cinque ontologie, verifica con HermiT e salva.
_qwen_host = os.environ.get("OLLAMA_HOST", "http://localhost:11434")
if not _qwen_host.startswith(("http://", "https://")):
    _qwen_host = "http://" + _qwen_host
_qwen_available = transitive_batch.api(_qwen_host, "/api/tags")["models"]
_qwen_names = [m["name"] for m in _qwen_available
               if m["name"].lower().startswith("qwen") and "cloud" not in m["name"].lower()]

def _qwen_version_key(name):
    match = re.search(r"qwen(\d+(?:\.\d+)*)", name.lower())
    version = tuple(int(part) for part in match.group(1).split(".")) if match else ()
    return version, name.endswith(":latest"), name

if QWEN_PREFERRED_MODEL in _qwen_names:
    QWEN_BATCH_MODEL = QWEN_PREFERRED_MODEL
elif _qwen_names:
    QWEN_BATCH_MODEL = max(_qwen_names, key=_qwen_version_key)
else:
    raise ValueError("Nessun modello Qwen locale disponibile: non viene effettuato alcun download.")
_qwen_slug = re.sub(r"[^A-Za-z0-9._-]+", "_", QWEN_BATCH_MODEL)
_qwen_batch_id = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ") + "_" + _qwen_slug + "_" + uuid.uuid4().hex[:8]
QWEN_BATCH_DIR = QWEN_BATCH_ROOT / "generated" / "transitive_property_ontologies" / _qwen_batch_id
print("Modello generatore:", QWEN_BATCH_MODEL, flush=True)
QWEN_BATCH_DIR, QWEN_BATCH_MANIFEST = transitive_batch.generate_ontology_batch(
    QWEN_BATCH_CONFIGS, QWEN_BATCH_MODEL, QWEN_BATCH_DIR,
    options=QWEN_BATCH_OPTIONS, think=False,
    system_prompt=QWEN_BATCH_SYSTEM_PROMPT,
    on_progress=lambda text: print(text, flush=True),
)
print("Batch salvato:", QWEN_BATCH_DIR, flush=True)


Modello generatore: qwen3.6:latest
Cinque ontologie generate, verificate con HermiT e salvate.
Batch salvato: /Users/emanuele/Desktop/neurosymbolic-epistemic/neurosymbolic-epistemic/generated/transitive_property_ontologies/20261001T153626Z_qwen3.6_latest_4f32ef54


In [ ]:
# Riepilogo: legge il manifest, senza chiamare il modello.
from IPython.display import Markdown, display
_qwen_rows = QWEN_BATCH_MANIFEST["items"]
_qwen_table = "| # | Dominio | Seed | Profondità | Distrattori | Triple originali | Verifica ritiro |\n|---|---|---:|---:|---:|---:|---|"
for item in _qwen_rows:
    verified = item["verification"]["original_entailed"] and not item["verification"]["withdrawal_entailed"]
    _qwen_table += (f"\n| {item['index']} | {item['domain']} | {item['seed']} | "
                    f"{item['path_length']} | {item['n_distractor_triples']} | "
                    f"{item['n_original_triples']} | {verified} |")
display(Markdown(_qwen_table))
print("Indice:", QWEN_BATCH_DIR / "batch.json")
for item in _qwen_rows:
    print(item["domain"], "—", QWEN_BATCH_DIR / item["directory"] / "base.ttl")


### Batch effettivamente generato e salvato

Modello locale: **qwen3.6:latest**. Cinque ontologie e cinque coppie verificate da HermiT.
Sono stati richiesti lo stesso numero di classi/proprietà dello schema e gli stessi parametri
per tutte le ontologie. I totali di triple differiscono leggermente per gli assiomi dello schema.

| # | Dominio | Profondità | Distrattori | Triple originali | Esito |
|---|---|---:|---:|---:|---|
| 1 | food | 3 | 10 | 62 | entailed → not_entailed |
| 2 | school | 3 | 10 | 61 | entailed → not_entailed |
| 3 | animals | 3 | 10 | 62 | entailed → not_entailed |
| 4 | relatives | 3 | 10 | 63 | entailed → not_entailed |
| 5 | food | 3 | 10 | 63 | entailed → not_entailed |

Directory: `generated/transitive_property_ontologies/20261001T153626Z_qwen3.6_latest_4f32ef54/`.

Ogni sottocartella contiene `base.ttl` (schema originale dell'LLM), `generation.json`
(provenienza e risposta integrale), `generation_attempts/` (anche risposte scartate)
e `dataset/` (query, snapshot originale e ritiro, hash e verifica).
Un tentativo preliminare fallito è conservato separatamente; il batch sopra è completo.
Non sono stati eseguiti i due turni di valutazione dell'LLM.
La cella seguente rilegge i risultati esistenti senza chiamare modelli o reasoner.


In [ ]:
# SOLA LETTURA del batch salvato: eseguibile senza generare altre ontologie.
from pathlib import Path
import json
from IPython.display import Markdown, display
_saved_roots = [Path.cwd(), Path.cwd() / "neurosymbolic-epistemic", Path.cwd().parent]
_saved_root = next((p.resolve() for p in _saved_roots
                    if (p / "transitive_property_benchmark.py").exists()), None)
if _saved_root is None:
    raise FileNotFoundError("Aprire il notebook dalla repository.")
QWEN_SAVED_BATCH_DIR = _saved_root / "generated/transitive_property_ontologies/20261001T153626Z_qwen3.6_latest_4f32ef54"
QWEN_SAVED_BATCH = json.loads((QWEN_SAVED_BATCH_DIR / "batch.json").read_text(encoding="utf-8"))
print("Stato:", QWEN_SAVED_BATCH["status"], "| Ontologie:", len(QWEN_SAVED_BATCH["items"]))
for item in QWEN_SAVED_BATCH["items"]:
    print(item["domain"], "—", QWEN_SAVED_BATCH_DIR / item["directory"] / "base.ttl")
